In [ ]:
# Install required packages
!pip install -q groq neo4j pymupdf sentence-transformers tqdm langchain-text-splitters

# Verify installations
import sys
print(f"Python version: {sys.version}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 50.3 MB/s eta 0:00:00
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import time
import re
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, asdict
from datetime import datetime
from google.colab import drive, userdata
from tqdm.notebook import tqdm
import pandas as pd

# External APIs
from groq import Groq
from neo4j import GraphDatabase
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer

# Mount Google Drive for persistence
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/research_knowledge_graph'
os.makedirs(f"{BASE_PATH}/papers", exist_ok=True)
os.makedirs(f"{BASE_PATH}/exports", exist_ok=True)

print(f"Working directory: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/research_knowledge_graph


In [ ]:
# Setup Groq API (Get from https://console.groq.com/keys)
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✓ Groq API key loaded from Colab secrets")
except:
    GROQ_API_KEY = input("Enter your Groq API Key: ")

# Initialize Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# Test Groq connection
try:
    test_response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Say 'Connected'"}],
        max_tokens=10
    )
    print(f"✓ Groq Connected: {test_response.choices[0].message.content}")
except Exception as e:
    print(f"✗ Groq Connection Failed: {e}")

✓ Groq API key loaded from Colab secrets
✓ Groq Connected: Connected


In [ ]:
# Neo4j AuraDB Credentials (Get from Neo4j Aura Console)
try:
    NEO4J_URI = userdata.get('NEO4J_URI')  # Format: neo4j+s://xxxx.databases.neo4j.io
    NEO4J_USER = userdata.get('NEO4J_USER', 'neo4j')
    NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')
    print("✓ Neo4j credentials loaded from secrets")
except:
    NEO4J_URI = input("Neo4j Aura URI: ")
    NEO4J_USER = input("Username (default: neo4j): ") or "neo4j"
    NEO4J_PASSWORD = input("Password: ")

# Initialize driver
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("✓ Neo4j AuraDB Connected Successfully")

    # Get database info
    with driver.session() as session:
        result = session.run("CALL dbms.components() YIELD name, versions RETURN name, versions")
        for record in result:
            print(f"  Database: {record['name']} v{record['versions'][0]}")
except Exception as e:
    print(f"✗ Neo4j Connection Failed: {e}")
    driver = None

Neo4j Aura URI: neo4j+s://7d0a734f.databases.neo4j.io
Username (default: neo4j): 7d0a734f
Password: eoRJp0s2_w_hRrQ1d51guV0M1qYMSDmkKtB4OLmX6TY
✓ Neo4j AuraDB Connected Successfully
  Database: Neo4j Kernel v5.27-aura
  Database: Cypher v5


In [ ]:
# Load lightweight embedding model (runs on CPU, ~22MB)
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Test embedding
test_emb = embedding_model.encode("This is a test sentence")
print(f"✓ Embedding model ready. Vector dimension: {len(test_emb)}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedding model ready. Vector dimension: 384


In [ ]:
def setup_schema(tx):
    """Create constraints and indexes for the knowledge graph"""
    # Constraints for uniqueness
    constraints = [
        "CREATE CONSTRAINT paper_id IF NOT EXISTS FOR (p:Paper) REQUIRE p.id IS UNIQUE",
        "CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (e:Entity) REQUIRE e.canonical_name IS UNIQUE",
        "CREATE CONSTRAINT author_name IF NOT EXISTS FOR (a:Author) REQUIRE a.name IS UNIQUE"
    ]

    # Indexes for performance
    indexes = [
        "CREATE INDEX paper_title IF NOT EXISTS FOR (p:Paper) ON (p.title)",
        "CREATE INDEX entity_type IF NOT EXISTS FOR (e:Entity) ON (e.type)",
        "CREATE INDEX rel_type IF NOT EXISTS FOR ()-[r:RELATES_TO]-() ON (r.relation_type)"
    ]

    for constraint in constraints:
        try:
            tx.run(constraint)
        except Exception as e:
            print(f"Constraint warning (may already exist): {e}")

    for index in indexes:
        try:
            tx.run(index)
        except Exception as e:
            print(f"Index warning: {e}")

if driver:
    with driver.session() as session:
        session.execute_write(setup_schema)
    print("✓ Schema constraints and indexes created")

✓ Schema constraints and indexes created


In [ ]:
class PaperProcessor:
    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.paper_id = os.path.basename(pdf_path).replace('.pdf', '')[:50]  # Truncate long names
        self.metadata = {}
        self.sections = {}

    def extract_text(self) -> Dict[str, Any]:
        """Extract structured text from PDF with improved section detection"""
        doc = fitz.open(self.pdf_path)

        # Extract metadata
        self.metadata = {
            'title': doc.metadata.get('title', self.paper_id),
            'author': doc.metadata.get('author', 'Unknown'),
            'paper_id': self.paper_id,
            'page_count': len(doc)
        }

        # Extract full text by pages
        pages_text = []
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text()
            pages_text.append(text)

        full_text = "\n".join(pages_text)
        doc.close()

        # Clean up headers/footers (journal noise)
        cleaned_text = self._clean_journal_noise(full_text)

        # Try section detection, fall back to chunking if fails
        sections = self._segment_sections_improved(cleaned_text)

        # If no sections found, create artificial ones
        if len(sections) <= 1:
            sections = self._fallback_chunking(cleaned_text)

        return {
            'metadata': self.metadata,
            'sections': sections,
            'full_text': cleaned_text[:50000]
        }

    def _clean_journal_noise(self, text: str) -> str:
        """Remove journal headers, footers, page numbers, ISSN lines"""
        lines = text.split('\n')
        cleaned_lines = []

        skip_patterns = [
            r'ISSN:', r'Vol\.\s*\d+', r'Issue\s*\d+', r'PP\.\s*\d+',
            r'https?://doi\.org/', r'Attribution-NonCommercial',
            r'Scientific Journal of', r'Artificial Intelligence and Blockchain',
            r'^\s*\d+\s*$',  # Page numbers alone
            r'Date of (Submission|Acceptance|Publication)',
            r'^\s*17\s*$|^\s*18\s*$|^\s*25\s*$'  # Specific page numbers from example
        ]

        for line in lines:
            line_stripped = line.strip()
            if not line_stripped:
                continue

            # Skip if matches any pattern
            skip = False
            for pattern in skip_patterns:
                if re.search(pattern, line_stripped, re.IGNORECASE):
                    skip = True
                    break

            # Skip lines that are just numbers (page numbers)
            if line_stripped.isdigit():
                skip = True

            if not skip:
                cleaned_lines.append(line)

        return '\n'.join(cleaned_lines)

    def _segment_sections_improved(self, text: str) -> Dict[str, str]:
        """Improved section detection with multiple strategies"""
        sections = {}

        # Strategy 1: Look for ALL CAPS section headers (common in academic papers)
        section_patterns = [
            (r'\bABSTRACT\b', 'abstract'),
            (r'\bINTRODUCTION\b', 'introduction'),
            (r'\bRELATED\s*WORK\b', 'related_work'),
            (r'\bMETHODOLOGY\b', 'methodology'),
            (r'\bMETHODS\b', 'methodology'),
            (r'\bPROPOSED\s*METHOD\b', 'methodology'),
            (r'\bEXPERIMENTS\b', 'experiments'),
            (r'\bEXPERIMENTAL\s*SETUP\b', 'experiments'),
            (r'\bEVALUATION\b', 'experiments'),
            (r'\bRESULTS\b', 'results'),
            (r'\bDISCUSSION\b', 'discussion'),
            (r'\bCONCLUSION\b', 'conclusion'),
            (r'\bCONCLUSIONS\b', 'conclusion'),
            (r'\bFUTURE\s*WORK\b', 'future_work'),
            (r'\bREFERENCES\b', 'references')
        ]

        # Find all section positions
        section_positions = []
        for pattern, section_name in section_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                section_positions.append((match.start(), section_name, match.group()))

        # Sort by position
        section_positions.sort()

        # Extract text between sections
        if section_positions:
            # Add preamble if first section doesn't start at beginning
            if section_positions[0][0] > 100:
                sections['preamble'] = text[:section_positions[0][0]].strip()

            # Extract each section
            for i, (start_pos, section_name, matched_text) in enumerate(section_positions):
                # Find end (start of next section or end of text)
                if i < len(section_positions) - 1:
                    end_pos = section_positions[i + 1][0]
                else:
                    end_pos = len(text)

                # Extract content after the header
                header_end = start_pos + len(matched_text)
                content = text[header_end:end_pos].strip()

                # Clean up: remove just the header, keep the rest
                # Remove leading punctuation and whitespace
                content = re.sub(r'^[\s\.\,\-\:]+', '', content)

                if len(content) > 50:  # Only add if substantial content
                    if section_name in sections:
                        # If section already exists, append (handling duplicates)
                        sections[section_name] += '\n' + content
                    else:
                        sections[section_name] = content
        else:
            # Strategy 2: Try numbered sections (1. Introduction, 2. Methodology)
            numbered_pattern = r'(?:\n|\r|^)\s*(\d+\.?\s+[A-Z][A-Za-z\s]+)\s*\n'
            matches = list(re.finditer(numbered_pattern, text))

            if matches:
                for i, match in enumerate(matches):
                    section_title = match.group(1).strip()
                    start = match.end()
                    end = matches[i+1].start() if i < len(matches)-1 else len(text)
                    content = text[start:end].strip()

                    # Normalize section name
                    section_key = section_title.lower().replace(' ', '_').replace('.', '')[:30]
                    sections[section_key] = content

        return sections

    def _fallback_chunking(self, text: str) -> Dict[str, str]:
        """If section detection fails, split into semantic chunks"""
        sections = {}

        # Try to extract abstract manually (usually first 200-500 words after "Abstract")
        abstract_match = re.search(r'(?i)abstract\s*[:.]?\s*(.*?)(?:\n\s*\n|\n\s*1\.|\n\s*introduction)', text, re.DOTALL)
        if abstract_match:
            sections['abstract'] = abstract_match.group(1).strip()[:2000]

        # Split rest into beginning, middle, end
        words = text.split()
        total_words = len(words)

        if total_words > 1000:
            sections['introduction'] = ' '.join(words[:500])
            sections['methodology'] = ' '.join(words[500:max(1000, total_words-500)])
            sections['conclusion'] = ' '.join(words[max(1000, total_words-500):])
        else:
            sections['full_text'] = text

        return sections

# Test the fixed processor immediately
print("Testing fixed processor...")
if pdf_files:
    test_path = f"{BASE_PATH}/papers/{pdf_files[0]}"
    processor = PaperProcessor(test_path)
    paper_data = processor.extract_text()

    print(f"\n✅ FIXED - Sections found: {list(paper_data['sections'].keys())}")
    for section, content in paper_data['sections'].items():
        print(f"  • {section}: {len(content)} chars")

    # Show abstract preview
    if 'abstract' in paper_data['sections']:
        print(f"\n📄 Abstract preview:\n{paper_data['sections']['abstract'][:400]}...")
else:
    print("No PDF files found to test")

Testing fixed processor...


NameError: name 'pdf_files' is not defined

In [ ]:
class KnowledgeExtractor:
    def __init__(self, client: Groq, model: str = "llama-3.3-70b-versatile"):
        self.client = client
        self.model = model
        self.rate_limit_delay = 1  # seconds between calls

    def extract_entities(self, text: str, context: str = "general") -> List[Dict]:
        """
        Extract entities from text using Groq
        Returns list of: {name, type, description, confidence}
        """
        prompt = f"""Analyze this research paper text and extract key entities.

Context: This is a {context} research paper.
Text excerpt:
{text[:4000]}

Extract entities from these categories:
1. CONCEPT (theories, algorithms, models, frameworks)
2. METHOD (techniques, approaches, procedures)
3. TOOL (software, datasets, platforms)
4. METRIC (evaluation measures, benchmarks)
5. PROBLEM (challenges, issues addressed)

Return ONLY a JSON array in this format:
[
  {{
    "name": "Exact entity name",
    "type": "CONCEPT|METHOD|TOOL|METRIC|PROBLEM",
    "description": "Brief description from context",
    "confidence": 0.95
  }}
]

Rules:
- Normalize names (e.g., "CNNs" → "Convolutional Neural Network")
- Include specific variants mentioned (e.g., "ResNet-50")
- Confidence 0.0-1.0 based on clarity in text
- Return empty array [] if no entities found"""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                response_format={"type": "json_object"}
            )

            time.sleep(self.rate_limit_delay)  # Rate limiting

            result = json.loads(response.choices[0].message.content)
            return result if isinstance(result, list) else result.get('entities', [])

        except Exception as e:
            print(f"Extraction error: {e}")
            return []

    def extract_relationships(self, text: str, entities: List[Dict]) -> List[Dict]:
        """
        Extract relationships between identified entities
        Returns: {source, target, relation_type, evidence, strength}
        """
        entity_names = [e['name'] for e in entities]

        prompt = f"""Given this text and the entities mentioned, identify relationships between them.

Text: {text[:4000]}

Entities mentioned: {entity_names}

Identify relationships like:
- IMPLEMENTS (Method implements Concept)
- USES (Method/Concept uses Tool/Dataset)
- IMPROVES_UPON (Method improves Method)
- EVALUATED_ON (Method evaluated on Dataset/Metric)
- ADDRESSES (Method addresses Problem)
- COMPARES_TO (Method compared to Method)

Return JSON array:
[
  {{
    "source": "Entity name",
    "target": "Entity name",
    "relation_type": "IMPLEMENTS|USES|IMPROVES_UPON|EVALUATED_ON|ADDRESSES|COMPARES_TO",
    "evidence": "Quote from text showing this relationship",
    "strength": 0.9
  }}
]"""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                response_format={"type": "json_object"}
            )

            time.sleep(self.rate_limit_delay)

            result = json.loads(response.choices[0].message.content)
            return result if isinstance(result, list) else result.get('relationships', [])

        except Exception as e:
            print(f"Relationship extraction error: {e}")
            return []

# Initialize extractor
extractor = KnowledgeExtractor(groq_client)
print("✓ Knowledge Extractor initialized")

✓ Knowledge Extractor initialized


In [ ]:
class GraphIngestor:
    def __init__(self, driver):
        self.driver = driver
        self.embedding_model = embedding_model

    def ingest_paper(self, paper_data: Dict):
        """Ingest paper metadata and content"""
        meta = paper_data['metadata']

        # Generate embedding for paper abstract/title
        text_for_emb = f"{meta.get('title', '')} {paper_data.get('abstract', '')}"
        embedding = self.embedding_model.encode(text_for_emb).tolist()

        with self.driver.session() as session:
            session.run("""
                MERGE (p:Paper {id: $id})
                SET p.title = $title,
                    p.author = $author,
                    p.page_count = $page_count,
                    p.processed_date = $date,
                    p.embedding = $embedding
            """, {
                'id': meta['paper_id'],
                'title': meta.get('title', 'Unknown'),
                'author': meta.get('author', 'Unknown'),
                'page_count': meta.get('page_count', 0),
                'date': datetime.now().isoformat(),
                'embedding': embedding
            })

    def ingest_entity(self, entity: Dict, paper_id: str):
        """Ingest single entity with embedding"""
        name = entity['name']
        entity_type = entity['type']
        description = entity.get('description', '')

        # Create canonical ID (lowercase, alphanumeric)
        canonical_id = re.sub(r'[^a-z0-9]', '_', name.lower())

        # Generate embedding for semantic search
        emb_text = f"{name}: {description}"
        embedding = self.embedding_model.encode(emb_text).tolist()

        with self.driver.session() as session:
            session.run("""
                MERGE (e:Entity {canonical_name: $canonical_id})
                SET e.name = $name,
                    e.type = $type,
                    e.description = $description,
                    e.embedding = $embedding,
                    e.first_seen_paper = $paper_id,
                    e.aliases = CASE
                        WHEN e.aliases IS NULL THEN [$name]
                        ELSE apoc.coll.toSet(e.aliases + [$name])
                    END
            """, {
                'canonical_id': canonical_id,
                'name': name,
                'type': entity_type,
                'description': description[:500],
                'embedding': embedding,
                'paper_id': paper_id
            })

            # Link to paper
            session.run("""
                MATCH (p:Paper {id: $paper_id})
                MATCH (e:Entity {canonical_name: $entity_id})
                MERGE (p)-[:MENTIONS {confidence: $confidence}]->(e)
            """, {
                'paper_id': paper_id,
                'entity_id': canonical_id,
                'confidence': entity.get('confidence', 0.8)
            })

        return canonical_id

    def ingest_relationship(self, rel: Dict, paper_id: str):
        """Ingest relationship between entities"""
        source_id = re.sub(r'[^a-z0-9]', '_', rel['source'].lower())
        target_id = re.sub(r'[^a-z0-9]', '_', rel['target'].lower())

        with self.driver.session() as session:
            session.run("""
                MATCH (s:Entity {canonical_name: $source_id})
                MATCH (t:Entity {canonical_name: $target_id})
                MERGE (s)-[r:RELATES_TO {
                    relation_type: $rel_type,
                    paper_id: $paper_id,
                    evidence: $evidence,
                    strength: $strength
                }]->(t)
            """, {
                'source_id': source_id,
                'target_id': target_id,
                'rel_type': rel['relation_type'],
                'paper_id': paper_id,
                'evidence': rel.get('evidence', '')[:500],
                'strength': rel.get('strength', 0.8)
            })

# Initialize ingestor
if driver:
    ingestor = GraphIngestor(driver)
    print("✓ Graph Ingestor ready")

✓ Graph Ingestor ready


In [ ]:
def process_papers_pipeline(pdf_dir: str):
    """
    Main pipeline: PDF → Text → Entities → Relationships → Neo4j
    """
    pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith('.pdf')]
    print(f"Starting processing of {len(pdf_files)} papers...\n")

    stats = {
        'processed': 0,
        'entities': 0,
        'relationships': 0,
        'errors': []
    }

    for pdf_file in tqdm(pdf_files, desc="Processing papers"):
        try:
            pdf_path = os.path.join(pdf_dir, pdf_file)
            paper_id = pdf_file.replace('.pdf', '')

            # Step 1: Extract text
            processor = PaperProcessor(pdf_path)
            paper_data = processor.extract_text()

            # Step 2: Ingest paper node
            ingestor.ingest_paper(paper_data)

            # Step 3: Extract entities from key sections
            all_entities = []
            sections_to_process = ['abstract', 'methodology', 'introduction']

            for section_name in sections_to_process:
                if section_name in paper_data['sections']:
                    section_text = paper_data['sections'][section_name]
                    entities = extractor.extract_entities(
                        section_text,
                        context=paper_data['metadata'].get('title', 'research')
                    )
                    all_entities.extend(entities)

            # Deduplicate entities by name
            seen = set()
            unique_entities = []
            for e in all_entities:
                if e['name'].lower() not in seen and e.get('confidence', 0) > 0.7:
                    seen.add(e['name'].lower())
                    unique_entities.append(e)

            # Step 4: Ingest entities
            entity_ids = []
            for entity in unique_entities:
                eid = ingestor.ingest_entity(entity, paper_id)
                entity_ids.append(eid)

            # Step 5: Extract and ingest relationships
            # Use methodology section for relationships
            if 'methodology' in paper_data['sections']:
                rels = extractor.extract_relationships(
                    paper_data['sections']['methodology'],
                    unique_entities
                )

                for rel in rels:
                    if rel.get('strength', 0) > 0.6:  # Filter weak relationships
                        ingestor.ingest_relationship(rel, paper_id)
                        stats['relationships'] += 1

            stats['processed'] += 1
            stats['entities'] += len(unique_entities)

            # Save checkpoint every 2 papers
            if stats['processed'] % 2 == 0:
                print(f"  Checkpoint: {stats['processed']} papers, {stats['entities']} entities")

        except Exception as e:
            error_msg = f"Error processing {pdf_file}: {str(e)}"
            print(f"✗ {error_msg}")
            stats['errors'].append(error_msg)
            continue

    return stats

# Run the pipeline (Make sure papers are in the folder first!)
# Upload PDFs to /content/drive/MyDrive/research_knowledge_graph/papers/ before running
if driver and pdf_files:
    results = process_papers_pipeline(f"{BASE_PATH}/papers")
    print(f"\n{'='*50}")
    print("PROCESSING COMPLETE")
    print(f"{'='*50}")
    print(f"Papers processed: {results['processed']}")
    print(f"Total entities: {results['entities']}")
    print(f"Total relationships: {results['relationships']}")
    if results['errors']:
        print(f"Errors: {len(results['errors'])}")
else:
    print("⚠️ No PDFs found or Neo4j not connected. Upload PDFs to Drive folder first.")

Starting processing of 5 papers...



Processing papers:   0%|          | 0/5 [00:00<?, ?it/s]

  Checkpoint: 2 papers, 41 entities


  Checkpoint: 4 papers, 86 entities



PROCESSING COMPLETE
Papers processed: 5
Total entities: 112
Total relationships: 44


=== PDF EXTRACTION TEST ===
Paper ID: Autonomous_Drone_Navigation_Using_Reinforcement_Le
Title: 
Sections found: ['preamble']
❌ No 'abstract' section found! Check section detection.


In [ ]:
def validate_graph():
    """Run validation queries to check graph health"""
    if not driver:
        print("Neo4j not connected")
        return

    queries = {
        "Papers in graph": "MATCH (p:Paper) RETURN count(p) as count",
        "Total entities": "MATCH (e:Entity) RETURN count(e) as count",
        "Entity types distribution": "MATCH (e:Entity) RETURN e.type as type, count(e) as count ORDER BY count DESC",
        "Total relationships": "MATCH ()-[r:RELATES_TO]->() RETURN count(r) as count",
        "Isolated entities (no relations)": "MATCH (e:Entity) WHERE NOT (e)-[:RELATES_TO]-() RETURN count(e) as count",
        "Most connected entities": """
            MATCH (e:Entity)-[r:RELATES_TO]-()
            RETURN e.name as entity, e.type as type, count(r) as connections
            ORDER BY connections DESC LIMIT 10
        """,
        "Sample paths": """
            MATCH path = (e1:Entity)-[r:RELATES_TO*1..2]->(e2:Entity)
            RETURN path LIMIT 5
        """
    }

    print("GRAPH VALIDATION RESULTS\n" + "="*50)

    with driver.session() as session:
        for name, query in queries.items():
            try:
                result = session.run(query)
                records = list(result)
                print(f"\n📊 {name}:")
                for record in records:
                    print(f"  {record}")
            except Exception as e:
                print(f"\n⚠️ {name}: Error - {e}")

# Run validation
validate_graph()

# Export to JSON backup
def export_graph():
    """Export graph to JSON for backup"""
    with driver.session() as session:
        # Export nodes
        nodes = session.run("""
            MATCH (n)
            WHERE n:Paper OR n:Entity
            RETURN labels(n) as labels, properties(n) as props, id(n) as nid
        """).data()

        # Export relationships
        rels = session.run("""
            MATCH ()-[r:RELATES_TO|MENTIONS]->()
            RETURN type(r) as type, properties(r) as props,
                   id(startNode(r)) as source, id(endNode(r)) as target
        """).data()

        export_data = {
            'exported_at': datetime.now().isoformat(),
            'nodes': nodes,
            'relationships': rels
        }

        export_path = f"{BASE_PATH}/exports/graph_backup_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
        with open(export_path, 'w') as f:
            json.dump(export_data, f, indent=2)

        print(f"✓ Graph exported to: {export_path}")
        return export_path

if driver:
    export_graph()

GRAPH VALIDATION RESULTS

📊 Papers in graph:
  <Record count=5>

📊 Total entities:
  <Record count=90>

📊 Entity types distribution:
  <Record type='CONCEPT' count=36>
  <Record type='PROBLEM' count=23>
  <Record type='METHOD' count=20>
  <Record type='TOOL' count=10>
  <Record type='METRIC' count=1>

📊 Total relationships:
  <Record count=38>

📊 Isolated entities (no relations):
  <Record count=52>

📊 Most connected entities:
  <Record entity='Reinforcement Learning' type='CONCEPT' connections=24>
  <Record entity='Autonomous Navigation' type='METHOD' connections=4>
  <Record entity='Proximal Policy Optimization' type='CONCEPT' connections=3>
  <Record entity='Adaptive Planning' type='CONCEPT' connections=3>
  <Record entity='Deep Deterministic Policy Gradient' type='CONCEPT' connections=3>
  <Record entity='Traditional Navigation' type='METHOD' connections=2>
  <Record entity='Kalman Filter' type='CONCEPT' connections=2>
  <Record entity='LIDAR' type='TOOL' connections=2>
  <Record e

✓ Graph exported to: /content/drive/MyDrive/research_knowledge_graph/exports/graph_backup_20260319_1942.json


In [ ]:
# Example queries to explore your graph after processing

def query_graph(question_type: str):
    """Run example queries"""
    with driver.session() as session:
        if question_type == "concepts":
            # What concepts are mentioned across multiple papers?
            result = session.run("""
                MATCH (p:Paper)-[:MENTIONS]->(e:Entity {type: 'CONCEPT'})
                WITH e, count(p) as papers
                WHERE papers > 1
                RETURN e.name as concept, papers, e.description
                ORDER BY papers DESC
            """)
            return list(result)

        elif question_type == "connections":
            # Find paths between methods and tools
            result = session.run("""
                MATCH path = (m:Entity {type: 'METHOD'})-[:RELATES_TO*1..3]->(t:Entity {type: 'TOOL'})
                RETURN path LIMIT 5
            """)
            return list(result)

        elif question_type == "problems":
            # What problems are addressed by which methods?
            result = session.run("""
                MATCH (m:Entity {type: 'METHOD'})-[r:RELATES_TO {relation_type: 'ADDRESSES'}]->(p:Entity {type: 'PROBLEM'})
                RETURN m.name as method, p.name as problem, r.evidence as evidence
            """).data()
            return result

# Uncomment to test after processing:
# concepts = query_graph("concepts")
# print(f"Found {len(concepts)} concepts mentioned in multiple papers")

In [ ]:
def query_graph_rag(question: str):
    """
    Retrieve context from Neo4j and synthesize answer with Groq
    """
    print(f"❓ Question: {question}\n")

    # Step 1: Find relevant entities in the graph (simple keyword match for now)
    keywords = [word for word in question.lower().split() if len(word) > 3]

    with driver.session() as session:
        # Find entities matching keywords
        result = session.run("""
            MATCH (e:Entity)
            WHERE ANY(keyword IN $keywords WHERE toLower(e.name) CONTAINS keyword)
            RETURN e.name as name, e.type as type, e.description as desc
            LIMIT 5
        """, {'keywords': keywords})

        entities = list(result)

        if not entities:
            print("⚠️ No relevant entities found in graph")
            return

        print(f"📍 Found {len(entities)} relevant entities:")
        entity_names = [e['name'] for e in entities]
        for e in entities:
            print(f"   • {e['name']} ({e['type']})")

        # Step 2: Get neighborhood (1-hop connections)
        result = session.run("""
            MATCH (e:Entity)-[r:RELATES_TO]-(neighbor:Entity)
            WHERE e.name IN $entity_names
            RETURN e.name as source, r.relation_type as rel,
                   neighbor.name as target, r.evidence as evidence
            LIMIT 10
        """, {'entity_names': entity_names})

        relationships = list(result)
        print(f"\n🔗 Found {len(relationships)} relationships:")
        for r in relationships:
            print(f"   • {r['source']} → [{r['rel']}] → {r['target']}")

    # Step 3: Synthesize with LLM
    context = "Knowledge Graph Context:\n"
    context += "Entities:\n" + "\n".join([f"- {e['name']}: {e['desc'][:100]}" for e in entities])
    context += "\n\nRelationships:\n" + "\n".join([f"- {r['source']} {r['rel']} {r['target']}" for r in relationships])

    prompt = f"""Answer the question using ONLY the provided knowledge graph context.
If the context doesn't contain enough information, say so.

{context}

Question: {question}

Answer:"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=500
        )

        print(f"\n🤖 LLM Answer:\n{'='*50}")
        print(response.choices[0].message.content)
        print(f"{'='*50}")

    except Exception as e:
        print(f"Error: {e}")

# Test it!
query_graph_rag("What methods address obstacle avoidance in drone navigation?")
# Or try: query_graph_rag("How does reinforcement learning compare to traditional navigation?")

❓ Question: What methods address obstacle avoidance in drone navigation?

📍 Found 5 relevant entities:
   • Autonomous Drone Navigation (METHOD)
   • Dynamic Obstacles (PROBLEM)
   • Drone Navigation Task (PROBLEM)
   • Collision Avoidance (PROBLEM)
   • Obstacle Avoidance (PROBLEM)

🔗 Found 6 relationships:
   • Autonomous Drone Navigation → [ADDRESSES] → Dynamic Environment
   • Dynamic Obstacles → [ADDRESSES] → Adaptive Planning
   • Drone Navigation Task → [IMPLEMENTS] → Reinforcement Learning
   • Collision Avoidance → [ADDRESSES] → Reinforcement Learning
   • Obstacle Avoidance → [ADDRESSES] → Reinforcement Learning
   • Obstacle Avoidance → [ADDRESSES] → Reinforcement Learning

🤖 LLM Answer:
Based on the provided knowledge graph context, the methods that address obstacle avoidance in drone navigation are:

1. Reinforcement Learning (as Obstacle Avoidance ADDRESSES Reinforcement Learning)
2. Autonomous Drone Navigation (as it ADDRESSES Dynamic Environment, which includes obstacle

In [ ]:
# Install pyvis for visualization
!pip install -q pyvis

from pyvis.network import Network
import networkx as nx

def visualize_subgraph():
    """Create interactive HTML visualization"""

    # Query Neo4j for subgraph (top connected nodes only for clarity)
    with driver.session() as session:
        # Get top 20 most connected entities and their neighbors
        result = session.run("""
            MATCH (e:Entity)-[r:RELATES_TO]-(e2:Entity)
            WITH e, count(r) as conn
            ORDER BY conn DESC
            LIMIT 15
            MATCH (e)-[r:RELATES_TO]-(neighbor:Entity)
            RETURN e.name as node1, e.type as type1,
                   r.relation_type as rel,
                   neighbor.name as node2, neighbor.type as type2
        """)

        edges = list(result)

    if not edges:
        print("No relationships to visualize")
        return

    # Create PyVis network
    net = Network(height="600px", width="100%", bgcolor="#ffffff", font_color="black")
    net.barnes_hut()

    # Color scheme by type
    colors = {
        'CONCEPT': '#4285F4',    # Blue
        'METHOD': '#34A853',     # Green
        'TOOL': '#FBBC04',       # Yellow
        'PROBLEM': '#EA4335',    # Red
        'METRIC': '#9AA0A6'      # Gray
    }

    added_nodes = set()

    for edge in edges:
        # Add nodes
        for node, node_type in [(edge['node1'], edge['type1']), (edge['node2'], edge['type2'])]:
            if node not in added_nodes:
                net.add_node(
                    node,
                    label=node,
                    color=colors.get(node_type, '#999999'),
                    title=f"Type: {node_type}",
                    size=20 if node == 'Reinforcement Learning' else 15
                )
                added_nodes.add(node)

        # Add edge
        net.add_edge(
            edge['node1'],
            edge['node2'],
            title=edge['rel'],
            label=edge['rel'][:10]  # Shorten label
        )

    # Save and display
    output_path = "/content/drive/MyDrive/research_knowledge_graph/graph_visualization.html"
    net.save_graph(output_path)

    # Display in iframe
    from IPython.display import IFrame, display
    display(IFrame(output_path, width="100%", height="600px"))

    print(f"✅ Interactive graph saved to: {output_path}")
    print("💡 Tip: Hover over nodes to see types, over edges to see relationships")

# Generate visualization
visualize_subgraph()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.8 MB/s eta 0:00:00


✅ Interactive graph saved to: /content/drive/MyDrive/research_knowledge_graph/graph_visualization.html
💡 Tip: Hover over nodes to see types, over edges to see relationships


# 🚀 DAY 2: AGENTIC INTELLIGENCE LAYER
**Date:** 2026-03-21  
**Goal:** Build LangGraph workflow on top of yesterday's Knowledge Graph  
**Prerequisites:** All Day 1 cells above must have run successfully


In [ ]:
# DAY 2 HEALTH CHECK - Run this first
print("🔍 VERIFYING YESTERDAY'S WORK...")

# 1. Check Drive still mounted
import os
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive mounted")
    backup_files = [f for f in os.listdir(f"{BASE_PATH}/exports") if f.endswith('.json')]
    print(f"✅ Found {len(backup_files)} backup(s): {backup_files[-1] if backup_files else 'None'}")
else:
    print("❌ Drive not mounted - run: drive.mount('/content/drive')")

# 2. Check Neo4j connection
try:
    driver.verify_connectivity()
    with driver.session() as session:
        result = session.run("MATCH (e:Entity) RETURN count(e) as cnt").single()
        print(f"✅ Neo4j connected: {result['cnt']} entities available")
except Exception as e:
    print(f"❌ Neo4j issue: {e}")

# 3. Check Groq
try:
    test = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=5
    )
    print("✅ Groq API responsive")
except Exception as e:
    print(f"❌ Groq issue: {e}")

print("\n🎯 Ready for Day 2 Agentic Layer!")

🔍 VERIFYING YESTERDAY'S WORK...
✅ Drive mounted
✅ Found 1 backup(s): graph_backup_20260319_1942.json
✅ Neo4j connected: 90 entities available
✅ Groq API responsive

🎯 Ready for Day 2 Agentic Layer!


In [ ]:
# Fix OpenTelemetry conflict
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp opentelemetry-instrumentation -q
!pip install -q opentelemetry-api==1.25.0 opentelemetry-sdk==1.25.0
!pip install -q chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 15.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.40.0 requires opentelemetry-sdk~=1.40.0, but you have opentelemetry-sdk 1.25.0 which is incompatible.
opentelemetry-resourcedetector-gcp 1.11.0a0 requires opentelemetry-api~=1.30, but you have opentelemetry-api 1.25.0 which is incompatible.
opentelemetry-resourcedetector-gcp 1.11.0a0 requires opentelemetry-sdk~=1.30, but you have opentelemetry-sdk 1.25.0 which is incompatible.
opentelemetry-exporter-gcp-trace 1.11.0 requires opentelemetry-api~=1.30, but you have opentelemetry-api 1.25.0 which is incompatible.
opentelemetry-exporter-gcp-tra

In [ ]:
# Install ChromaDB

import chromadb
from chromadb.config import Settings
import uuid

# Initialize ChromaDB with persistence in Drive
chroma_path = f"{BASE_PATH}/chroma_db"
os.makedirs(chroma_path, exist_ok=True)

chroma_client = chromadb.PersistentClient(
    path=chroma_path,
    settings=Settings(anonymized_telemetry=False)
)

# Create collection for paper chunks
collection = chroma_client.get_or_create_collection(
    name="research_papers",
    metadata={"description": "Drone navigation research paper chunks"}
)

print(f"✅ ChromaDB initialized at: {chroma_path}")
print(f"📊 Existing collections: {chroma_client.list_collections()}")

✅ ChromaDB initialized at: /content/drive/MyDrive/research_knowledge_graph/chroma_db
📊 Existing collections: [Collection(name=research_papers)]


In [ ]:
# QUICK RECOVERY - Reload PaperProcessor and continue ChromaDB

# 1. Verify imports are available
import os
import re
import json
import fitz  # PyMuPDF
from typing import Dict, Any, List
import uuid

# 2. Re-define PaperProcessor (from yesterday)
class PaperProcessor:
    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.paper_id = os.path.basename(pdf_path).replace('.pdf', '')[:50]
        self.metadata = {}
        self.sections = {}

    def extract_text(self) -> Dict[str, Any]:
        doc = fitz.open(self.pdf_path)
        self.metadata = {
            'title': doc.metadata.get('title', self.paper_id),
            'author': doc.metadata.get('author', 'Unknown'),
            'paper_id': self.paper_id,
            'page_count': len(doc)
        }

        pages_text = []
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text()
            pages_text.append(text)

        full_text = "\n".join(pages_text)
        doc.close()
        cleaned_text = self._clean_journal_noise(full_text)
        sections = self._segment_sections_improved(cleaned_text)

        if len(sections) <= 1:
            sections = self._fallback_chunking(cleaned_text)

        return {
            'metadata': self.metadata,
            'sections': sections,
            'full_text': cleaned_text[:50000]
        }

    def _clean_journal_noise(self, text: str) -> str:
        lines = text.split('\n')
        cleaned_lines = []
        skip_patterns = [
            r'ISSN:', r'Vol\.\s*\d+', r'Issue\s*\d+', r'PP\.\s*\d+',
            r'https?://doi\.org/', r'Attribution-NonCommercial',
            r'Scientific Journal of', r'Artificial Intelligence and Blockchain',
            r'^\s*\d+\s*$', r'Date of (Submission|Acceptance|Publication)',
            r'^\s*1\d\s*$|^\s*2\d\s*$|^\s*\d{1,2}\s*$'
        ]

        for line in lines:
            line_stripped = line.strip()
            if not line_stripped:
                continue
            skip = any(re.search(p, line_stripped, re.IGNORECASE) for p in skip_patterns)
            if line_stripped.isdigit():
                skip = True
            if not skip:
                cleaned_lines.append(line)

        return '\n'.join(cleaned_lines)

    def _segment_sections_improved(self, text: str) -> Dict[str, str]:
        sections = {}
        section_patterns = [
            (r'\bABSTRACT\b', 'abstract'), (r'\bINTRODUCTION\b', 'introduction'),
            (r'\bRELATED\s*WORK\b', 'related_work'), (r'\bMETHODOLOGY\b', 'methodology'),
            (r'\bMETHODS\b', 'methodology'), (r'\bPROPOSED\s*METHOD\b', 'methodology'),
            (r'\bEXPERIMENTS\b', 'experiments'), (r'\bEVALUATION\b', 'experiments'),
            (r'\bRESULTS\b', 'results'), (r'\bCONCLUSION\b', 'conclusion'),
            (r'\bREFERENCES\b', 'references')
        ]

        section_positions = []
        for pattern, section_name in section_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                section_positions.append((match.start(), section_name, match.group()))

        section_positions.sort()

        if section_positions:
            if section_positions[0][0] > 100:
                sections['preamble'] = text[:section_positions[0][0]].strip()

            for i, (start_pos, section_name, matched_text) in enumerate(section_positions):
                if i < len(section_positions) - 1:
                    end_pos = section_positions[i + 1][0]
                else:
                    end_pos = len(text)

                header_end = start_pos + len(matched_text)
                content = text[header_end:end_pos].strip()
                content = re.sub(r'^[\s\.\,\-\:]+', '', content)

                if len(content) > 50:
                    if section_name in sections:
                        sections[section_name] += '\n' + content
                    else:
                        sections[section_name] = content
        else:
            sections = self._fallback_chunking(text)

        return sections

    def _fallback_chunking(self, text: str) -> Dict[str, str]:
        sections = {}
        abstract_match = re.search(r'(?i)abstract\s*[:.]?\s*(.*?)(?:\n\s*\n|\n\s*1\.|\n\s*introduction)', text, re.DOTALL)
        if abstract_match:
            sections['abstract'] = abstract_match.group(1).strip()[:2000]

        words = text.split()
        total_words = len(words)
        if total_words > 1000:
            sections['introduction'] = ' '.join(words[:500])
            sections['methodology'] = ' '.join(words[500:max(1000, total_words-500)])
            sections['conclusion'] = ' '.join(words[max(1000, total_words-500):])
        else:
            sections['full_text'] = text

        return sections

# 3. Re-import ChromaDB
import chromadb
from chromadb.config import Settings

# 4. Verify paths
BASE_PATH = '/content/drive/MyDrive/research_knowledge_graph'
chroma_path = f"{BASE_PATH}/chroma_db"

chroma_client = chromadb.PersistentClient(
    path=chroma_path,
    settings=Settings(anonymized_telemetry=False)
)
collection = chroma_client.get_or_create_collection(name="research_papers")

print("✅ Recovery complete! PaperProcessor reloaded")
print(f"📊 ChromaDB has {collection.count()} existing chunks")

✅ Recovery complete! PaperProcessor reloaded
📊 ChromaDB has 0 existing chunks


In [ ]:
def populate_chroma_from_papers():
    """
    Load paper sections into ChromaDB for vector search
    Uses yesterday's extracted text
    """
    # Check if already populated
    if collection.count() > 0:
        print(f"⚠️ ChromaDB already has {collection.count()} chunks. Skip? (y/n)")
        return

    pdf_files = [f for f in os.listdir(f"{BASE_PATH}/papers") if f.endswith('.pdf')]
    print(f"Processing {len(pdf_files)} papers into ChromaDB...")

    all_chunks = []

    for pdf_file in pdf_files:
        try:
            # Reuse yesterday's processor
            pdf_path = os.path.join(f"{BASE_PATH}/papers", pdf_file)
            processor = PaperProcessor(pdf_path)
            paper_data = processor.extract_text()

            paper_id = paper_data['metadata']['paper_id']

            # Chunk each section separately
            for section_name, section_text in paper_data['sections'].items():
                if len(section_text.strip()) < 50:
                    continue

                # Split long sections into smaller chunks (500 chars overlap)
                words = section_text.split()
                chunk_size = 200  # words
                overlap = 50      # words

                for i in range(0, len(words), chunk_size - overlap):
                    chunk_words = words[i:i + chunk_size]
                    chunk_text = ' '.join(chunk_words)

                    if len(chunk_text) > 100:
                        all_chunks.append({
                            'id': str(uuid.uuid4()),
                            'text': chunk_text,
                            'metadata': {
                                'paper_id': paper_id,
                                'section': section_name,
                                'chunk_index': i // (chunk_size - overlap),
                                'source_file': pdf_file
                            }
                        })

            print(f"  ✅ {pdf_file}: {len([c for c in all_chunks if c['metadata']['paper_id'] == paper_id])} chunks")

        except Exception as e:
            print(f"  ❌ Error with {pdf_file}: {e}")

    # Batch insert into ChromaDB
    if all_chunks:
        batch_size = 100
        for i in range(0, len(all_chunks), batch_size):
            batch = all_chunks[i:i+batch_size]
            collection.add(
                ids=[c['id'] for c in batch],
                documents=[c['text'] for c in batch],
                metadatas=[c['metadata'] for c in batch]
            )

        print(f"\n🎯 Successfully added {len(all_chunks)} chunks to ChromaDB")
    else:
        print("⚠️ No chunks generated")

# Run population
populate_chroma_from_papers()

Processing 5 papers into ChromaDB...
  ✅ Autonomous_Drone_Navigation_Using_Reinforcement_Le.pdf: 15 chunks
  ✅ Reinforcement_Learning_in_Autonomous_Navigation_Ov.pdf: 44 chunks
  ✅ Dynamic_Navigation_in_Unconstrained_Environments_u.pdf: 108 chunks
  ✅ Beyond_Static_Obstacles_Integrating_Kalman_Filter_.pdf: 63 chunks
  ✅ Cost-Effective_Autonomous_Drone_Navigation_Using_R.pdf: 63 chunks


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 28.6MiB/s]



🎯 Successfully added 293 chunks to ChromaDB


In [ ]:
def test_vector_search(query: str, n_results: int = 3):
    """Quick test of ChromaDB functionality"""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )

    print(f"🔍 Query: '{query}'\n")
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        print(f"Result {i+1} (distance: {dist:.3f}):")
        print(f"  Paper: {meta['paper_id']}")
        print(f"  Section: {meta['section']}")
        print(f"  Preview: {doc[:150]}...\n")

# Test it
test_vector_search("reinforcement learning obstacle avoidance", n_results=3)

🔍 Query: 'reinforcement learning obstacle avoidance'

Result 1 (distance: 0.624):
  Paper: Reinforcement_Learning_in_Autonomous_Navigation_Ov
  Section: methodology
  Preview: in complex scenarios where obstacles are moving, or the terrain is irregular. Another approach uses grid- based path planning, where the environment i...

Result 2 (distance: 0.659):
  Paper: Reinforcement_Learning_in_Autonomous_Navigation_Ov
  Section: methodology
  Preview: which can be learned independently. This hierarchical structure allows the agent to focus on specific aspects of the navigation task, such as obstacle...

Result 3 (distance: 0.681):
  Paper: Reinforcement_Learning_in_Autonomous_Navigation_Ov
  Section: methodology
  Preview: the expected return of taking a particular action in a given state. This dual approach allows for more stable and efficient learning, as the actor con...



In [ ]:
from typing import List, Dict, Any, Optional, Literal
from dataclasses import dataclass, field

@dataclass
class TrajectoryStep:
    """Single step in agent reasoning for observability"""
    step_number: int
    node_name: str
    action: str
    input_summary: str
    output_summary: str
    timestamp: str

@dataclass
class GraphPath:
    """Represents a path through knowledge graph"""
    nodes: List[str]
    relationships: List[str]
    evidence: List[str]
    path_length: int

@dataclass
class AgentState:
    """
    State object passed between LangGraph nodes
    This is the 'memory' of your agent
    """
    # Inputs
    question: str = ""

    # Planning
    strategy: Literal["vector_only", "graph_only", "hybrid"] = "hybrid"
    plan: List[str] = field(default_factory=list)

    # Retrieved Context
    vector_context: List[Dict] = field(default_factory=list)
    graph_paths: List[GraphPath] = field(default_factory=list)

    # Audit Trail (for UI visualization tomorrow)
    trajectory: List[TrajectoryStep] = field(default_factory=list)
    token_usage: int = 0

    # Output
    synthesis: str = ""
    confidence: float = 0.0
    citations: List[str] = field(default_factory=list)

    def add_trajectory(self, node: str, action: str, input_s: str, output_s: str):
        """Helper to log agent steps"""
        from datetime import datetime
        self.trajectory.append(TrajectoryStep(
            step_number=len(self.trajectory) + 1,
            node_name=node,
            action=action,
            input_summary=input_s[:100],
            output_summary=output_s[:100],
            timestamp=datetime.now().isoformat()
        ))

    def to_dict(self) -> Dict:
        """Convert for LangGraph compatibility"""
        return {
            'question': self.question,
            'strategy': self.strategy,
            'plan': self.plan,
            'vector_context': self.vector_context,
            'graph_paths': [self._path_to_dict(p) for p in self.graph_paths],
            'trajectory': [self._traj_to_dict(t) for t in self.trajectory],
            'token_usage': self.token_usage,
            'synthesis': self.synthesis,
            'confidence': self.confidence,
            'citations': self.citations
        }

    def _path_to_dict(self, p: GraphPath) -> Dict:
        return {
            'nodes': p.nodes,
            'relationships': p.relationships,
            'path_length': p.path_length
        }

    def _traj_to_dict(self, t: TrajectoryStep) -> Dict:
        return {
            'step': t.step_number,
            'node': t.node_name,
            'action': t.action
        }

print("✅ AgentState class defined")
print("📋 Example structure:")
example = AgentState(question="How does RL work?")
example.add_trajectory("Planner", "strategy_selected", "question", "hybrid")
print(f"   Trajectory: {example.trajectory[0].action}")

✅ AgentState class defined
📋 Example structure:
   Trajectory: strategy_selected


Tools Implementaion


In [ ]:
def tool_vector_search(query: str, top_k: int = 5, paper_filter: List[str] = None) -> List[Dict]:
    """
    Retrieve relevant text chunks from ChromaDB
    Returns list of dicts with text, metadata, and similarity score
    """
    try:
        # Build filter if specified
        where_filter = None
        if paper_filter:
            where_filter = {"paper_id": {"$in": paper_filter}}

        # Query ChromaDB
        results = collection.query(
            query_texts=[query],
            n_results=top_k,
            where=where_filter,
            include=['documents', 'metadatas', 'distances']
        )

        # Format output
        formatted_results = []
        for i in range(len(results['documents'][0])):
            formatted_results.append({
                'text': results['documents'][0][i],
                'paper_id': results['metadatas'][0][i]['paper_id'],
                'section': results['metadatas'][0][i]['section'],
                'chunk_index': results['metadatas'][0][i]['chunk_index'],
                'similarity_score': 1 - results['distances'][0][i]  # Convert distance to similarity
            })

        return formatted_results

    except Exception as e:
        print(f"Vector search error: {e}")
        return []

# Test it
test_vector_results = tool_vector_search("proximal policy optimization obstacle avoidance", top_k=3)
print(f"Found {len(test_vector_results)} chunks")
for r in test_vector_results[:2]:
    print(f"  - {r['paper_id']} [{r['section']}] (score: {r['similarity_score']:.3f})")

Found 3 chunks
  - Dynamic_Navigation_in_Unconstrained_Environments_u [experiments] (score: 0.503)
  - Reinforcement_Learning_in_Autonomous_Navigation_Ov [methodology] (score: 0.301)


In [ ]:
def tool_graph_lookup(entity_name: str, relation_type: str = None) -> List[Dict]:
    """
    Find direct neighbors of an entity in Neo4j (1-hop)
    Good for: "What is X?" or "What relates to X?"
    """
    try:
        with driver.session() as session:
            # Build query
            if relation_type:
                query = """
                MATCH (e:Entity {name: $entity_name})-[r:RELATES_TO {relation_type: $rel_type}]-(neighbor:Entity)
                RETURN neighbor.name as target, r.relation_type as relation,
                       r.evidence as evidence, r.strength as strength
                LIMIT 10
                """
                result = session.run(query, {
                    'entity_name': entity_name,
                    'rel_type': relation_type
                })
            else:
                query = """
                MATCH (e:Entity)-[r:RELATES_TO]-(neighbor:Entity)
                WHERE toLower(e.name) CONTAINS toLower($entity_name)
                RETURN e.name as source, neighbor.name as target,
                       r.relation_type as relation, r.evidence as evidence,
                       r.strength as strength
                LIMIT 15
                """
                result = session.run(query, {'entity_name': entity_name})

            return [dict(record) for record in result]

    except Exception as e:
        print(f"Graph lookup error: {e}")
        return []

# Test it
test_graph_results = tool_graph_lookup("Reinforcement Learning")
print(f"Found {len(test_graph_results)} direct relationships")
for r in test_graph_results[:3]:
    print(f"  - {r.get('source', 'N/A')} → [{r['relation']}] → {r['target']}")

Found 15 direct relationships
  - Reinforcement Learning → [IMPLEMENTS] → Autonomous Navigation
  - Reinforcement Learning → [IMPROVES_UPON] → Model Predictive Control
  - Reinforcement Learning → [USES] → LIDAR


In [ ]:
def tool_multihop_paths(start_entity: str, end_entity: str = None, max_hops: int = 3) -> List[GraphPath]:
    """
    Find paths through knowledge graph (2-3 hops)
    Fixed: Using f-string for hop range since Neo4j doesn't parameterize depth
    """
    paths = []

    try:
        with driver.session() as session:
            if end_entity:
                # Specific path finding (A to B) - FIXED with f-string for hops
                query = f"""
                MATCH path = (start:Entity)-[r:RELATES_TO*1..{max_hops}]-(end:Entity)
                WHERE toLower(start.name) CONTAINS toLower($start_name)
                  AND toLower(end.name) CONTAINS toLower($end_name)
                RETURN [n IN nodes(path) | n.name] as node_names,
                       [rel IN relationships(path) | rel.relation_type] as rel_types,
                       [rel IN relationships(path) | rel.evidence] as evidence_list,
                       length(path) as path_length
                ORDER BY path_length
                LIMIT 5
                """
                result = session.run(query, {
                    'start_name': start_entity,
                    'end_name': end_entity
                })
            else:
                # Exploration mode - FIXED with f-string for hops
                query = f"""
                MATCH path = (start:Entity)-[r:RELATES_TO*2..{max_hops}]-(end:Entity)
                WHERE toLower(start.name) CONTAINS toLower($start_name)
                  AND start <> end
                WITH path, nodes(path) as ns, relationships(path) as rels
                RETURN [n IN ns | n.name] as node_names,
                       [rel IN rels | rel.relation_type] as rel_types,
                       [rel IN rels | rel.evidence] as evidence_list,
                       length(path) as path_length
                ORDER BY path_length
                LIMIT 10
                """
                result = session.run(query, {'start_name': start_entity})

            for record in result:
                paths.append(GraphPath(
                    nodes=record['node_names'],
                    relationships=record['rel_types'],
                    evidence=record['evidence_list'],
                    path_length=record['path_length']
                ))

        return paths

    except Exception as e:
        print(f"Multi-hop error: {e}")
        import traceback
        traceback.print_exc()
        return []

# Test it again
test_paths = tool_multihop_paths("Reinforcement Learning", "Obstacle Avoidance", max_hops=3)
print(f"Found {len(test_paths)} paths from RL to Obstacle Avoidance")
for i, path in enumerate(test_paths[:3], 1):
    if path.nodes and path.relationships:
        path_str = " → ".join([f"{n} [{r}]" for n, r in zip(path.nodes[:-1], path.relationships)]) + f" → {path.nodes[-1]}"
        print(f"  Path {i} (length {path.path_length}): {path_str}")

Found 5 paths from RL to Obstacle Avoidance
  Path 1 (length 1): Reinforcement Learning [ADDRESSES] → Obstacle Avoidance
  Path 2 (length 1): Reinforcement Learning [ADDRESSES] → Obstacle Avoidance
  Path 3 (length 3): Reinforcement Learning [USES] → LIDAR [USES] → Reinforcement Learning [ADDRESSES] → Obstacle Avoidance


In [ ]:
# Quick recovery - only need Groq client
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
groq_client = Groq(api_key=GROQ_API_KEY)

# Verify
test = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Hi"}],
    max_tokens=5
)
print("✅ Groq client recovered!")

✅ Groq client recovered!


In [ ]:
def tool_compress_context(long_context: str, target_tokens: int = 4000) -> str:
    """
    Compress long context to fit token limits using iterative summarization
    Strategy: Extract key facts, drop redundant text
    """
    # Rough estimation: 1 token ≈ 4 characters
    current_chars = len(long_context)
    if current_chars < target_tokens * 4:
        return long_context  # No compression needed

    try:
        # If too long, use LLM to summarize
        compression_prompt = f"""Compress this research context while preserving:
- Key entities and their relationships
- Specific methods and their performance claims
- Citation sources

Original length: {len(long_context)} chars. Target: ~{target_tokens} tokens.

Context:
{long_context[:8000]}  # Limit input to avoid token overflow

Provide a dense summary (bullet points) preserving technical accuracy:"""

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": compression_prompt}],
            temperature=0.2,
            max_tokens=1500
        )

        compressed = response.choices[0].message.content
        print(f"  Compressed {current_chars} chars → {len(compressed)} chars")
        return compressed

    except Exception as e:
        print(f"Compression error: {e}")
        # Fallback: Truncate intelligently
        return long_context[:target_tokens * 4] + "... [truncated]"

# Test with dummy long text
test_compression = tool_compress_context(
    "Reinforcement Learning (RL) is a method. " * 100 +
    "PPO is an algorithm that addresses obstacle avoidance. " * 50,
    target_tokens=500
)
print(f"Compression test result length: {len(test_compression)} chars")

  Compressed 6850 chars → 1187 chars
Compression test result length: 1187 chars


In [ ]:
!pip install -q langgraph

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

print("✅ LangGraph ready")

✅ LangGraph ready


In [ ]:
def node_query_planner(state: AgentState) -> AgentState:
    """
    Analyzes question complexity and decides retrieval strategy
    Updates: strategy, plan, trajectory
    """
    question = state.question.lower()

    # LLM-based planning for complex decisions
    planning_prompt = f"""Analyze this research question and decide the best retrieval strategy.

Question: "{state.question}"

Available strategies:
- "vector_only": Simple factual lookup (what is X?, define Y)
- "graph_only": Relationship/path questions (how does X relate to Y?, which methods address Z?)
- "hybrid": Comparison or synthesis questions (compare X and Y, analyze trade-offs)

Respond with ONLY a JSON object:
{{"strategy": "vector_only|graph_only|hybrid", "plan": ["step1", "step2", "step3"], "reasoning": "brief explanation"}}

Strategy selection rules:
- "what is", "define", "explain" → vector_only
- "how does", "relate", "connect", "path" → graph_only
- "compare", "vs", "difference", "trade-off", "best" → hybrid"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": planning_prompt}],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        plan_data = json.loads(response.choices[0].message.content)
        state.strategy = plan_data.get('strategy', 'hybrid')
        state.plan = plan_data.get('plan', ['retrieve', 'synthesize'])

    except Exception as e:
        # Fallback to keyword-based
        if any(word in question for word in ['compare', 'vs', 'difference', 'better']):
            state.strategy = "hybrid"
            state.plan = ["vector_search", "graph_explore", "synthesize"]
        elif any(word in question for word in ['how', 'relate', 'connect', 'path']):
            state.strategy = "graph_only"
            state.plan = ["graph_explore", "synthesize"]
        else:
            state.strategy = "vector_only"
            state.plan = ["vector_search", "synthesize"]

    # Log to trajectory
    state.add_trajectory(
        node="QueryPlanner",
        action=f"Selected {state.strategy} strategy",
        input_s=state.question,
        output_s=str(state.plan)
    )

    print(f"🎯 Strategy: {state.strategy} | Plan: {state.plan}")
    return state

# Quick test
test_state = AgentState(question="Compare PPO and DDPG for obstacle avoidance")
test_state = node_query_planner(test_state)
print(f"Trajectory: {test_state.trajectory[0].action}")

🎯 Strategy: hybrid | Plan: ['Identify key characteristics of PPO and DDPG', 'Analyze performance metrics for obstacle avoidance in both algorithms', 'Compare trade-offs and advantages of each approach']
Trajectory: Selected hybrid strategy


In [ ]:
def node_context_retriever(state: AgentState) -> AgentState:
    """
    Retrieves relevant chunks from ChromaDB based on question
    Updates: vector_context, trajectory
    """
    if "vector_search" not in state.plan and state.strategy != "vector_only":
        state.add_trajectory("ContextRetriever", "skipped", "strategy is graph_only", "no action")
        return state

    # Enhance query with key entities if we know them
    search_query = state.question

    # Get results
    results = tool_vector_search(search_query, top_k=5)
    state.vector_context = results

    # Calculate token estimate (rough: 4 chars per token)
    total_chars = sum(len(r['text']) for r in results)
    state.token_usage += total_chars // 4

    state.add_trajectory(
        node="ContextRetriever",
        action=f"Retrieved {len(results)} chunks",
        input_s=search_query,
        output_s=f"From papers: {set(r['paper_id'] for r in results)}"
    )

    print(f"📚 Retrieved {len(results)} chunks from ChromaDB")
    return state

# Test
test_state = node_context_retriever(test_state)
print(f"Top chunk preview: {test_state.vector_context[0]['text'][:100]}..." if test_state.vector_context else "No chunks")

No chunks


In [ ]:
def node_graph_explorer(state: AgentState) -> AgentState:
    """
    Explores knowledge graph for multi-hop reasoning
    Updates: graph_paths, trajectory
    """
    if "graph_explore" not in state.plan and state.strategy == "vector_only":
        state.add_trajectory("GraphExplorer", "skipped", "strategy is vector_only", "no action")
        return state

    # Extract entities from question (simple keyword matching for now)
    # In production, use NER here
    question_words = state.question.split()
    potential_entities = [w for w in question_words if len(w) > 3 and w.lower() not in ['what', 'which', 'how', 'does', 'and', 'for', 'the', 'with']]

    all_paths = []

    # Try to find paths between pairs of entities mentioned
    if len(potential_entities) >= 2:
        for i in range(min(3, len(potential_entities))):
            for j in range(i+1, min(4, len(potential_entities))):
                paths = tool_multihop_paths(
                    potential_entities[i],
                    potential_entities[j],
                    max_hops=3
                )
                all_paths.extend(paths)

    # Also explore from single entities if no pairs
    if not all_paths and potential_entities:
        for entity in potential_entities[:2]:
            paths = tool_multihop_paths(entity, max_hops=3)
            all_paths.extend(paths)

    # Deduplicate paths
    seen_paths = set()
    unique_paths = []
    for path in all_paths:
        path_key = tuple(path.nodes)
        if path_key not in seen_paths:
            seen_paths.add(path_key)
            unique_paths.append(path)

    state.graph_paths = unique_paths[:5]  # Keep top 5

    state.add_trajectory(
        node="GraphExplorer",
        action=f"Found {len(unique_paths)} unique paths",
        input_s=str(potential_entities),
        output_s=f"Paths: {[p.nodes for p in unique_paths[:2]]}"
    )

    print(f"🕸️ Found {len(unique_paths)} graph paths")
    return state

# Test
test_state = node_graph_explorer(test_state)
print(f"Sample path: {' -> '.join(test_state.graph_paths[0].nodes) if test_state.graph_paths else 'No paths'}")

🕸️ Found 5 graph paths
Sample path: Twin Delayed DDPG -> Deep Deterministic Policy Gradient -> Reinforcement Learning -> Obstacle Avoidance


In [ ]:
def node_synthesizer(state: AgentState) -> AgentState:
    """
    Combines vector and graph context into final answer
    Updates: synthesis, confidence, citations
    """
    # Build context sections
    vector_section = ""
    if state.vector_context:
        vector_section = "RELEVANT TEXT CHUNKS:\n"
        for i, chunk in enumerate(state.vector_context, 1):
            vector_section += f"[{i}] From {chunk['paper_id']} ({chunk['section']}): {chunk['text'][:300]}...\n"

    graph_section = ""
    if state.graph_paths:
        graph_section = "\nKNOWLEDGE GRAPH PATHS:\n"
        for i, path in enumerate(state.graph_paths, 1):
            path_str = " → ".join([f"{n} [{r}]" for n, r in zip(path.nodes[:-1], path.relationships)]) + f" → {path.nodes[-1]}"
            graph_section += f"[{i}] {path_str}\n"

    # Construct prompt
    synthesis_prompt = f"""Answer the research question using the provided context. Be specific and cite sources.

Question: {state.question}

{vector_section}
{graph_section}

Instructions:
1. Answer directly based on the provided context
2. Cite specific papers/chunks using [reference] format
3. If comparing multiple approaches, be explicit about trade-offs
4. If graph paths show relationships, explain the reasoning chain
5. State confidence level (high/medium/low) based on evidence quality

Answer:"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": synthesis_prompt}],
            temperature=0.3,
            max_tokens=800
        )

        state.synthesis = response.choices[0].message.content

        # Extract confidence (simple regex)
        if "high confidence" in state.synthesis.lower():
            state.confidence = 0.9
        elif "medium confidence" in state.synthesis.lower():
            state.confidence = 0.7
        elif "low confidence" in state.synthesis.lower():
            state.confidence = 0.5
        else:
            state.confidence = 0.75  # Default

        # Extract citations
        import re
        state.citations = list(set(re.findall(r'\[(\d+)\]', state.synthesis)))

    except Exception as e:
        state.synthesis = f"Error generating answer: {e}"
        state.confidence = 0.0

    state.add_trajectory(
        node="Synthesizer",
        action="Generated answer",
        input_s=f"Used {len(state.vector_context)} chunks and {len(state.graph_paths)} paths",
        output_s=f"Answer length: {len(state.synthesis)} chars, Confidence: {state.confidence}"
    )

    print(f"✅ Answer generated (confidence: {state.confidence})")
    return state

# Test full pipeline
test_state = node_synthesizer(test_state)
print(f"\n🤖 ANSWER:\n{test_state.synthesis[:500]}...")

✅ Answer generated (confidence: 0.75)

🤖 ANSWER:
To compare PPO (Proximal Policy Optimization) and DDPG (Deep Deterministic Policy Gradient) for obstacle avoidance, we must consider the context provided by the knowledge graph paths. 

Firstly, DDPG is directly related to obstacle avoidance through its implementation of reinforcement learning [1, 2], which addresses both obstacle avoidance and collision avoidance. This indicates that DDPG is a viable approach for navigating through environments with obstacles.

On the other hand, PPO is not exp...


In [ ]:
# Define the workflow graph
workflow = StateGraph(AgentState)

# Add all nodes
workflow.add_node("planner", node_query_planner)
workflow.add_node("retriever", node_context_retriever)
workflow.add_node("explorer", node_graph_explorer)
workflow.add_node("synthesizer", node_synthesizer)

print("✅ Nodes added to workflow")

✅ Nodes added to workflow


In [ ]:
def route_by_strategy(state: AgentState) -> str:
    """
    Conditional routing based on planner's strategy decision
    Returns: next node name
    """
    if state.strategy == "vector_only":
        return "retriever"
    elif state.strategy == "graph_only":
        return "explorer"
    else:  # hybrid
        return "retriever"  # Will go to explorer after

def after_retriever(state: AgentState) -> str:
    """
    After retriever, either go to explorer (if hybrid) or synthesizer
    """
    if state.strategy == "hybrid" and "graph_explore" in state.plan:
        return "explorer"
    else:
        return "synthesizer"

def after_explorer(state: AgentState) -> str:
    """Always go to synthesizer after explorer"""
    return "synthesizer"

# Add edges
workflow.set_entry_point("planner")

# From planner: conditional routing
workflow.add_conditional_edges(
    "planner",
    route_by_strategy,
    {
        "retriever": "retriever",
        "explorer": "explorer"
    }
)

# From retriever: either to explorer (hybrid) or synthesizer
workflow.add_conditional_edges(
    "retriever",
    after_retriever,
    {
        "explorer": "explorer",
        "synthesizer": "synthesizer"
    }
)

# From explorer: always to synthesizer
workflow.add_conditional_edges(
    "explorer",
    after_explorer,
    {
        "synthesizer": "synthesizer"
    }
)

# From synthesizer: end
workflow.add_edge("synthesizer", END)

print("✅ Workflow edges configured")
print("Flow: planner → [retriever|explorer] → [explorer|synthesizer] → synthesizer → END")

✅ Workflow edges configured
Flow: planner → [retriever|explorer] → [explorer|synthesizer] → synthesizer → END


In [ ]:
# Compile the workflow
agent_app = workflow.compile()

print("🚀 Agent compiled successfully!")
print("Ready to answer questions...")

# Test function - FIXED for both dict state and dataclass trajectory
def ask_question(question: str) -> Dict:
    """
    Run a question through the agentic workflow
    """
    # Initialize state
    initial_state = AgentState(question=question)

    # Run workflow - returns dict
    final_state = agent_app.invoke(initial_state)

    # Handle trajectory (list of TrajectoryStep dataclasses)
    trajectory_list = []
    for t in final_state['trajectory']:
        trajectory_list.append({
            'step': t.step_number,  # Access as attribute, not dict key
            'node': t.node_name,
            'action': t.action
        })

    return {
        'question': final_state['question'],
        'strategy': final_state['strategy'],
        'answer': final_state['synthesis'],
        'confidence': final_state['confidence'],
        'trajectory': trajectory_list,
        'vector_chunks': len(final_state['vector_context']),
        'graph_paths': len(final_state['graph_paths'])
    }

# Test 1: Simple lookup
print("\n" + "="*60)
print("TEST 1: Simple Definition")
print("="*60)
result1 = ask_question("What is reinforcement learning?")
print(f"Strategy: {result1['strategy']}")
print(f"Answer: {result1['answer'][:300]}...")
print(f"Trajectory: {[t['node'] for t in result1['trajectory']]}")

# Test 2: Comparison (should use hybrid)
print("\n" + "="*60)
print("TEST 2: Comparison Question")
print("="*60)
result2 = ask_question("Compare PPO and DDPG for drone navigation")
print(f"Strategy: {result2['strategy']}")
print(f"Answer: {result2['answer'][:300]}...")
print(f"Trajectory: {[t['node'] for t in result2['trajectory']]}")
print(f"Graph paths used: {result2['graph_paths']}")

🚀 Agent compiled successfully!
Ready to answer questions...

TEST 1: Simple Definition
🎯 Strategy: vector_only | Plan: ['lookup definition', 'retrieve relevant information', 'provide factual answer']
📚 Retrieved 5 chunks from ChromaDB
✅ Answer generated (confidence: 0.75)
Strategy: vector_only
Answer: Reinforcement learning (RL) is a branch of machine learning where an agent learns to make decisions by interacting with its environment, based on the concept of trial and error [1]. The agent takes actions within the environment, receives feedback in the form of rewards or penalties, and updates its...
Trajectory: ['QueryPlanner', 'ContextRetriever', 'Synthesizer']

TEST 2: Comparison Question
🎯 Strategy: hybrid | Plan: ['Identify key characteristics of PPO and DDPG', 'Analyze performance metrics for drone navigation', 'Evaluate trade-offs between the two algorithms']
✅ Answer generated (confidence: 0.75)
Strategy: hybrid
Answer: When comparing Proximal Policy Optimization (PPO) and Deep 

In [ ]:
# Save the compiled agent and test results for tomorrow
import pickle
import json

# Save test fixtures (questions that work well)
test_fixtures = [
    "What is reinforcement learning?",
    "Compare PPO and DDPG for drone navigation",
    "Which methods address obstacle avoidance?",
    "How does reinforcement learning relate to LIDAR?"
]

checkpoint = {
    'date': '2026-03-21',
    'status': 'Agent core working',
    'test_fixtures': test_fixtures,
    'stats': {
        'entities_in_graph': 90,
        'relationships': 38,
        'chunks_in_chroma': collection.count(),
        'papers_processed': 5
    },
    'capabilities': [
        'Multi-hop graph traversal (2-3 hops)',
        'Strategy routing (vector/graph/hybrid)',
        'Context compression',
        'Trajectory logging'
    ]
}

checkpoint_path = f"{BASE_PATH}/agent_checkpoint_day2.json"
with open(checkpoint_path, 'w') as f:
    json.dump(checkpoint, f, indent=2)

print("✅ CHECKPOINT SAVED")
print(f"📁 Location: {checkpoint_path}")
print(f"🎯 Resume tomorrow with: Gradio UI (Hour 5)")

✅ CHECKPOINT SAVED
📁 Location: /content/drive/MyDrive/research_knowledge_graph/agent_checkpoint_day2.json
🎯 Resume tomorrow with: Gradio UI (Hour 5)


In [ ]:
!pip install -q gradio
import gradio as gr
print("✅ Gradio ready")

✅ Gradio ready


In [ ]:
def basic_rag_only(question: str):
    """Baseline: Vector search only (no graph)"""
    results = tool_vector_search(question, top_k=3)
    context = "\n\n".join([f"[{i+1}] {r['text'][:400]}..."
                          for i, r in enumerate(results)])

    prompt = f"""Answer based ONLY on these text chunks:
{context}

Question: {question}
Answer:"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500
    )

    sources = [f"{r['paper_id']} ({r['section']})" for r in results]
    return response.choices[0].message.content, "\n".join(sources)

def agentic_graph_rag(question: str):
    """Your innovation: Full agent with trajectory"""
    result = ask_question(question)

    # Format trajectory for display
    traj_display = []
    for t in result['trajectory']:
        emoji = {"QueryPlanner": "🎯", "ContextRetriever": "📚",
                "GraphExplorer": "🕸️", "Synthesizer": "📝"}.get(t['node'], "➡️")
        traj_display.append(f"{emoji} **{t['node']}**: {t['action']}")

    trajectory_md = "\n\n".join(traj_display)

    # Format graph paths
    paths_md = ""
    if result['graph_paths'] > 0:
        paths_md = f"\n\n**🔗 Graph Paths Used:** {result['graph_paths']} multi-hop connections explored"

    full_response = f"{result['answer']}\n\n---\n**Confidence:** {result['confidence']:.0%}{paths_md}"

    return full_response, trajectory_md, f"Strategy: {result['strategy']}"

# Build UI
with gr.Blocks(title="Agentic GraphRAG Research Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🔬 Research Knowledge Assistant
    ### Compare Basic RAG vs Agentic GraphRAG with Multi-Hop Reasoning
    """)

    with gr.Row():
        question_input = gr.Textbox(
            label="Research Question",
            placeholder="e.g., Compare PPO and DDPG for obstacle avoidance",
            lines=2,
            scale=3
        )
        submit_btn = gr.Button("Analyze", variant="primary", scale=1)

    with gr.Row():
        # Left: Basic RAG (Baseline)
        with gr.Column(scale=1):
            gr.Markdown("### 📚 Basic RAG (Vector Only)")
            basic_output = gr.Textbox(label="Answer", lines=10, show_copy_button=True)
            basic_sources = gr.Textbox(label="Sources", lines=3)

        # Right: Your Agentic System
        with gr.Column(scale=1):
            gr.Markdown("### 🕸️ Agentic GraphRAG (Multi-Hop)")
            agent_output = gr.Textbox(label="Answer", lines=10, show_copy_button=True)
            strategy_badge = gr.Label(label="Strategy Used")

            with gr.Accordion("🧠 View Reasoning Trajectory", open=False):
                trajectory_display = gr.Markdown()

    # Example buttons
    gr.Markdown("### 🎯 Example Questions")
    with gr.Row():
        ex1 = gr.Button("What is RL?", size="sm")
        ex2 = gr.Button("Compare PPO vs DDPG", size="sm")
        ex3 = gr.Button("Methods for obstacles?", size="sm")
        ex4 = gr.Button("How does RL relate to LIDAR?", size="sm")

    # Event handlers
    submit_btn.click(
        fn=basic_rag_only,
        inputs=question_input,
        outputs=[basic_output, basic_sources]
    )
    submit_btn.click(
        fn=agentic_graph_rag,
        inputs=question_input,
        outputs=[agent_output, trajectory_display, strategy_badge]
    )

    # Example handlers
    ex1.click(lambda: "What is reinforcement learning?", outputs=question_input)
    ex2.click(lambda: "Compare PPO and DDPG for drone navigation", outputs=question_input)
    ex3.click(lambda: "Which methods address obstacle avoidance?", outputs=question_input)
    ex4.click(lambda: "How does reinforcement learning relate to sensors?", outputs=question_input)

print("✅ UI built")

/tmp/ipykernel_10605/1423225370.py:46: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Agentic GraphRAG Research Assistant", theme=gr.themes.Soft()) as demo:


✅ UI built


In [ ]:
# Launch with public URL
print("🚀 Starting server...")
print("⏳ Generating public URL (may take 10 seconds)...")

demo.launch(
    share=True,  # This creates the public URL
    show_error=True,
    quiet=False
)

print("\n✅ DEPLOYED!")
print("🔗 Share this URL in your resume/README")

🚀 Starting server...
⏳ Generating public URL (may take 10 seconds)...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://451e96e72781a5733a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ DEPLOYED!
🔗 Share this URL in your resume/README


In [ ]:
!pip install chainlit groq neo4j chromadb pymupdf sentence-transformers langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-http to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.1 MB/

In [ ]:
import chainlit as cl
import os
import json
import re
import uuid
import asyncio
from datetime import datetime
from typing import List, Dict, Any, Optional, Literal
from dataclasses import dataclass, field

# External APIs
import fitz
from groq import Groq
from neo4j import GraphDatabase
import chromadb
from chromadb.config import Settings
from langgraph.graph import StateGraph, END

# ==========================================
# CONFIGURATION
# ==========================================
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
NEO4J_URI = os.environ.get("NEO4J_URI")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
NEO4J_USER = "neo4j"

# ==========================================
# PDF PROCESSOR
# ==========================================
class PaperProcessor:
    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.paper_id = os.path.basename(pdf_path).replace('.pdf', '')[:50]

    def extract_text(self):
        doc = fitz.open(self.pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()

        # Simple cleaning
        lines = text.split('\n')
        clean_lines = [l for l in lines if not any(x in l for x in ['ISSN:', 'Vol.', 'PP.', 'Date of'])]

        return {
            'metadata': {'paper_id': self.paper_id},
            'full_text': '\n'.join(clean_lines),
            'sections': {'full': text}
        }

# ==========================================
# KNOWLEDGE EXTRACTOR
# ==========================================
class KnowledgeExtractor:
    def __init__(self, client: Groq):
        self.client = client

    def extract_entities(self, text: str):
        if len(text) < 50:
            return []

        prompt = f"""Extract entities from this research text.
Categories: CONCEPT, METHOD, TOOL, PROBLEM
Return JSON: {{"entities": [{{"name": "...", "type": "CONCEPT", "confidence": 0.9}}]}}

Text: {text[:3000]}"""

        try:
            response = self.client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            data = json.loads(response.choices[0].message.content)
            return data.get('entities', [])
        except:
            return []

# ==========================================
# AGENT STATE
# ==========================================
@dataclass
class TrajectoryStep:
    step_number: int
    node_name: str
    action: str

@dataclass
class AgentState:
    question: str = ""
    strategy: str = "hybrid"
    plan: List[str] = field(default_factory=list)
    vector_context: List[Dict] = field(default_factory=list)
    graph_paths: List[Dict] = field(default_factory=list)
    trajectory: List[TrajectoryStep] = field(default_factory=list)
    synthesis: str = ""
    confidence: float = 0.0

    def add_trajectory(self, node: str, action: str):
        self.trajectory.append(TrajectoryStep(
            step_number=len(self.trajectory) + 1,
            node_name=node,
            action=action
        ))

# ==========================================
# TOOLS
# ==========================================
def tool_vector_search(query: str, collection, top_k: int = 5):
    try:
        results = collection.query(query_texts=[query], n_results=top_k)
        return [{'text': doc} for doc in results['documents'][0]]
    except:
        return []

def tool_graph_paths(query: str, driver):
    """Simplified graph search"""
    return []

# ==========================================
# AGENT NODES
# ==========================================
def node_planner(state: AgentState):
    q = state.question.lower()
    if any(x in q for x in ['compare', 'vs']):
        state.strategy, state.plan = "hybrid", ["vector", "graph", "synth"]
    elif any(x in q for x in ['how', 'which', 'relate']):
        state.strategy, state.plan = "graph", ["graph", "synth"]
    else:
        state.strategy, state.plan = "vector", ["vector", "synth"]

    state.add_trajectory("Planner", f"Strategy: {state.strategy}")
    return state

def node_retriever(state: AgentState, collection):
    results = tool_vector_search(state.question, collection)
    state.vector_context = results
    state.add_trajectory("Retriever", f"Found {len(results)} chunks")
    return state

def node_explorer(state: AgentState, driver):
    if state.strategy == "vector":
        state.add_trajectory("Explorer", "Skipped")
        return state
    paths = tool_graph_paths(state.question, driver)
    state.graph_paths = paths
    state.add_trajectory("Explorer", f"Found {len(paths)} paths")
    return state

def node_synthesizer(state: AgentState, groq_client):
    context = "\n".join([v['text'][:300] for v in state.vector_context[:3]])

    prompt = f"""Answer based on context:
{context}

Question: {state.question}
Answer:"""

    try:
        resp = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=600
        )
        state.synthesis = resp.choices[0].message.content
        state.confidence = 0.8
    except Exception as e:
        state.synthesis = f"Error: {str(e)}"

    state.add_trajectory("Synthesizer", "Generated answer")
    return state

# ==========================================
# CHAINLIT HANDLERS (No on_file_upload!)
# ==========================================
@cl.on_chat_start
async def start():
    session_id = str(uuid.uuid4())[:8]
    cl.user_session.set("id", session_id)
    cl.user_session.set("groq", Groq(api_key=GROQ_API_KEY))

    # Setup Chroma
    chroma_path = f"/tmp/chroma_{session_id}"
    client = chromadb.PersistentClient(path=chroma_path, settings=Settings(anonymized_telemetry=False))
    collection = client.get_or_create_collection(f"papers_{session_id}")
    cl.user_session.set("chroma", collection)

    cl.user_session.set("uploaded", False)

    await cl.Message(
        content="👋 **ResearchGPT Ready!**\n\n📎 **To upload a PDF**: Click the **attachment icon** (paperclip) below, select your PDF, then type 'process' and send.",
        author="System"
    ).send()

@cl.on_message
async def main(message: cl.Message):
    """Handles both file uploads and questions"""
    session_id = cl.user_session.get("id")
    collection = cl.user_session.get("chroma")
    groq = cl.user_session.get("groq")

    # Handle File Upload via elements (works in ALL versions)
    if message.elements and len(message.elements) > 0:
        for element in message.elements:
            if isinstance(element, cl.File):
                # Process PDF
                async with cl.Step(name="📄 Processing PDF...") as step:
                    temp_path = f"/tmp/{session_id}_{element.name}"
                    with open(temp_path, "wb") as f:
                        f.write(element.content)

                    step.output = f"Saved {element.name}"

                    # Extract
                    processor = PaperProcessor(temp_path)
                    paper_data = processor.extract_text()

                    step.name = "🔍 Extracting entities..."
                    await step.update()

                    extractor = KnowledgeExtractor(groq)
                    entities = extractor.extract_entities(paper_data['full_text'][:4000])

                    # Index
                    step.name = "📚 Indexing..."
                    await step.update()

                    words = paper_data['full_text'].split()
                    chunks = [' '.join(words[i:i+200]) for i in range(0, len(words), 150)]

                    if chunks:
                        collection.add(
                            ids=[f"{session_id}_{i}" for i in range(len(chunks))],
                            documents=chunks
                        )

                    cl.user_session.set("uploaded", True)
                    step.name = "✅ Ready!"

                await cl.Message(
                    content=f"🎯 **{element.name}** processed! Found {len(entities)} entities. Ask questions now.",
                    author="System"
                ).send()
                return

    # Handle "process" command or questions
    if not cl.user_session.get("uploaded"):
        await cl.Message(content="⚠️ Please upload a PDF first using the paperclip icon 📎, then type 'process'").send()
        return

    # Run Agent
    state = AgentState(question=message.content)

    # Workflow
    state = node_planner(state)
    state = node_retriever(state, collection)
    state = node_explorer(state, None)  # Neo4j driver if available
    state = node_synthesizer(state, groq)

    # Send response with streaming
    msg = cl.Message(content="")
    await msg.send()

    words = state.synthesis.split()
    for word in words:
        await msg.stream_token(word + " ")
        await asyncio.sleep(0.01)

    # Add reasoning
    traj = "\n".join([f"{t.step_number}. {t.node_name}: {t.action}" for t in state.trajectory])
    elements = [cl.Text(name="Reasoning", content=traj, display="side")]
    await cl.Message(content="", elements=elements, author="System").send()

if __name__ == "__main__":
    print("Run: chainlit run app.py -w")

Run: chainlit run app.py -w


In [ ]:
!pip install --upgrade chainlit -q

In [ ]:
!pip install -q chainlit pyngrok groq neo4j chromadb pymupdf sentence-transformers langgraph

In [ ]:
from pyngrok import ngrok
import os
from google.colab import userdata

# Get ngrok token (free from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# Kill existing tunnels
!killall ngrok 2>/dev/null || true

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import os

# Get ngrok token
try:
    ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(ngrok_token)
except:
    ngrok_token = input("Enter your ngrok token (get from https://dashboard.ngrok.com): ")
    ngrok.set_auth_token(ngrok_token)

# Get API keys
try:
    groq_key = userdata.get('GROQ_API_KEY')
    neo4j_uri = userdata.get('NEO4J_URI')
    neo4j_pass = userdata.get('NEO4J_PASSWORD')
except:
    groq_key = input("GROQ_API_KEY: ")
    neo4j_uri = input("NEO4J_URI (or press enter to skip): ")
    neo4j_pass = input("NEO4J_PASSWORD (or press enter to skip): ")

# Inject secrets into app.py
with open('/content/app.py', 'r') as f:
    content = f.read()

content = content.replace('PLACEHOLDER_GROQ', groq_key)
content = content.replace('PLACEHOLDER_NEO4J_URI', neo4j_uri if neo4j_uri else "neo4j://localhost")
content = content.replace('PLACEHOLDER_NEO4J_PASS', neo4j_pass if neo4j_pass else "password")

with open('/content/app.py', 'w') as f:
    f.write(content)

print("✅ API keys injected")

# Kill existing ngrok
!pkill -f ngrok 2>/dev/null || true

# Create tunnel
public_url = ngrok.connect(8000).public_url
print(f"🌐 Public URL: {public_url}")
print("⏳ Starting Chainlit server...")

# Run server
!cd /content && chainlit run app.py --port 8000 --headless

✅ API keys injected
^C
🌐 Public URL: https://6a6c-8-228-4-177.ngrok-free.app
⏳ Starting Chainlit server...
✅ app.py created successfully
Traceback (most recent call last):
  File "/usr/local/bin/chainlit", line 8, in <module>
    sys.exit(cli())
             ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1406, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1873, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1269, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-package